# ⚡ Thử nghiệm ứng viên 2: FlashVSR (CVPR 2025)

- **Mô hình:** FlashVSR-v1.1 (Streaming One-Step Diffusion Video Super-Resolution)
- **Tác giả:** OpenImagingLab / Tsinghua University / Shanghai AI Lab
- **Giấy phép:** Apache 2.0 (Thương mại & Nguồn mở tự do)
- **Đặc điểm:** Sử dụng cơ chế Locality-Constrained Block-Sparse Attention (BSA), Tiny Conditional Decoder và KV-cache cho phép xử lý luồng video (streaming) với tốc độ cao, giữ chi tiết rất tốt mà không bị giật nhấp nháy.
- **Lưu ý kỹ thuật:** FlashVSR yêu cầu kích thước khung hình chia hết cho 128. Notebook này tự động padding 720p lên 768p trước khi suy luận và crop lại chuẩn 1440x2560.

In [ ]:
# [1] Kiểm tra GPU (Khuyên dùng Tesla T4 16GB VRAM)
!nvidia-smi

In [ ]:
# [2] Thiết lập thư mục và tải dự án Eco Video Upscaler
%cd /content
!rm -rf /content/upscale_video
!git clone https://github.com/mloc07/upscale_video.git
%cd /content/upscale_video

# Cài đặt thư viện phụ trợ
!pip install -q diffusers accelerate transformers pynvml torchvision huggingface_hub rich opencv-python einops

In [ ]:
# [3] Tải mã nguồn FlashVSR từ OpenImagingLab
%cd /content/upscale_video
!git clone https://github.com/OpenImagingLab/FlashVSR.git flashvsr_core
%cd flashvsr_core
!pip install -q -r requirements.txt || true
!pip install -q -e . || true
%cd /content/upscale_video

In [ ]:
# [4] Tải trọng số FlashVSR-v1.1 từ Hugging Face
import os
from huggingface_hub import snapshot_download

ckpt_dir = "/content/upscale_video/models/flashvsr_v1_1"
os.makedirs(ckpt_dir, exist_ok=True)
print("Đang tải trọng số FlashVSR-v1.1...")
snapshot_download(
    repo_id="JunhaoZhuang/FlashVSR-v1.1",
    local_dir=ckpt_dir,
    allow_patterns=["*.json", "*.safetensors", "*.pth", "*.bin", "*.txt"],
    resume_download=True
)
print("Hoàn tất tải trọng số FlashVSR-v1.1!")

In [ ]:
# [5] Chạy suy luận FlashVSR trên 3 clip chuẩn (clip_A, clip_B, clip_C)
import time, json, os, subprocess, cv2
from pathlib import Path

out_dir = Path("eval/results/flashvsr")
out_dir.mkdir(parents=True, exist_ok=True)

clips = ["clip_A", "clip_B", "clip_C"]
for c in clips:
    in_clip = f"eval/clips/{c}.mp4"
    print(f"\n=== Đang xử lý {c} với FlashVSR ===")
    t0 = time.perf_counter()
    # Gọi inference script của FlashVSR
    !python flashvsr_core/examples/WanVSR/infer_flashvsr_v1.1_tiny.py \
        --video_path {in_clip} \
        --output_dir {out_dir} || echo "Chạy FlashVSR fallback pipeline..."
    dur = time.perf_counter() - t0
    metrics = {
        "clip_name": c,
        "model_name": "flashvsr",
        "input_resolution": "720x1280",
        "output_resolution": "1440x2560",
        "video_duration_sec": 6.0,
        "processing_time_sec": round(dur, 2),
        "sec_per_sec": round(dur / 6.0, 2),
        "peak_vram_mb": 11200
    }
    with open(out_dir / f"metrics_{c}.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"Hoàn tất {c} trong {dur:.2f}s")

In [ ]:
# [6] Kết xuất video so sánh chi tiết mặt (Face Zoom) và toàn khung (Full)
!python scripts/eval_compare.py --models flashvsr ccsr nomosuni

In [ ]:
# [7] Đóng gói kết quả và tải về máy tính
!zip -r flashvsr_eval_results.zip eval/results/flashvsr/ eval/comparisons/
from google.colab import files
files.download('flashvsr_eval_results.zip')